# FundusNet — Colab Training Pipeline

Train 5 models (Swin, MaxViT, ConvNeXt V2, EfficientNet V2, DeiT III) on GPU,
export to ONNX, and download for local deployment.

**Runtime:** GPU (T4) recommended. Go to `Runtime > Change runtime type > T4 GPU`.

**Dataset:** Upload `retina_dataset.zip` to your Google Drive before running.

## 1. Setup — Clone repo & install dependencies

In [ ]:
!git clone https://github.com/Mariakevin/FundusNet.git
%cd FundusNet

!pip install -r requirements.txt
!pip install timm onnxruntime

## 2. Mount Google Drive & load dataset

1. Upload `retina_dataset.zip` to your Google Drive (root or any folder)
2. Run the cell below — it will ask you to authorize access
3. Enter the path to your zip file (e.g. `/content/drive/MyDrive/retina_dataset.zip`)

In [ ]:
import os
import zipfile

from google.colab import drive

# Mount Google Drive
drive.mount("/content/drive")

# ---- UPDATE THIS PATH to where you put the zip in Google Drive ----
ZIP_PATH = "/content/drive/MyDrive/retina_dataset.zip"
# -------------------------------------------------------------------

if not os.path.exists(ZIP_PATH):
    print(f"ERROR: {ZIP_PATH} not found.")
    print("Upload retina_dataset.zip to Google Drive first, then update ZIP_PATH above.")
else:
    print(f"Found {ZIP_PATH} ({os.path.getsize(ZIP_PATH) / 1e6:.0f} MB)")
    print("Extracting...")
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(".")
    print("Done.")

# Verify dataset
dataset_path = "retina_dataset"
total = 0
for cls in ["Healthy", "Cataract", "Glaucoma", "Retina Disease"]:
    cls_path = os.path.join(dataset_path, cls)
    if os.path.exists(cls_path):
        count = len([f for f in os.listdir(cls_path) if f.lower().endswith((".png", ".jpg", ".jpeg"))])
        print(f"  {cls}: {count} images")
        total += count
    else:
        print(f"  {cls}: MISSING!")
print(f"Total: {total} images")

## 3. Verify GPU is available

In [ ]:
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("WARNING: No GPU. Go to Runtime > Change runtime type > T4 GPU")

## 4. Train All 5 Models

5-fold CV, 200 epochs each. ~4-6 hours on T4.

In [ ]:
!python train.py --models swin maxvit convnext_v2 efficientnet_v2 deit --epochs 200 --batch-size 32 --folds 5

## 5. Export to ONNX

In [ ]:
models = ["swin", "maxvit", "convnext_v2", "efficientnet_v2", "deit"]

for m in models:
    ckpt = f"experiments/models/{m}_best.pth"
    out = f"models/{m}_retinopathy.onnx"
    print(f"\nExporting {m}...")
    !python export_onnx.py --model {m} --checkpoint {ckpt} --output {out} --verify --quantize

## 6. Verify Exported Models

In [ ]:
import os

print("Exported ONNX models:")
for f in sorted(os.listdir("models")):
    if f.endswith(".onnx"):
        size_mb = os.path.getsize(os.path.join("models", f)) / 1e6
        print(f"  {f}: {size_mb:.1f} MB")

## 7. Quick Inference Test

In [ ]:
import time

import numpy as np
import onnxruntime as ort

categories = ["Healthy", "Cataract", "Glaucoma", "Retina Disease"]

for m in models:
    path = f"models/{m}_retinopathy.onnx"
    if not os.path.exists(path):
        print(f"  {m}: NOT FOUND")
        continue
    sess = ort.InferenceSession(path, providers=["CUDAExecutionProvider", "CPUExecutionProvider"])
    dummy = np.random.randn(1, 3, 224, 224).astype(np.float32)
    start = time.time()
    for _ in range(10):
        out = sess.run(None, {"input": dummy})
    ms = (time.time() - start) / 10 * 1000
    probs = np.exp(out[0][0]) / np.exp(out[0][0]).sum()
    print(f"  {m}: {categories[np.argmax(probs)]} ({max(probs) * 100:.1f}%) — {ms:.1f}ms")

## 8. Save trained models to Google Drive

In [ ]:
import shutil

# Create a folder in Drive for the models
drive_models = "/content/drive/MyDrive/FundusNet_models"
os.makedirs(drive_models, exist_ok=True)

# Copy ONNX models
for f in os.listdir("models"):
    if f.endswith(".onnx"):
        shutil.copy2(os.path.join("models", f), drive_models)
        print(f"  Copied {f}")

# Copy PyTorch checkpoints
drive_checkpoints = "/content/drive/MyDrive/FundusNet_checkpoints"
os.makedirs(drive_checkpoints, exist_ok=True)
for f in os.listdir("experiments/models"):
    if f.endswith(".pth"):
        shutil.copy2(os.path.join("experiments/models", f), drive_checkpoints)
        print(f"  Copied {f}")

print("\nSaved to Google Drive:")
print(f"  ONNX models:    {drive_models}/")
print(f"  Checkpoints:    {drive_checkpoints}/")
print("\nDownload from drive.google.com to your local models/ folder.")